<div style="border:1px solid #d0d0d0; padding:25px; border-radius:8px; background:#f8f9fa">

<h1 style="margin-bottom:10px">
Corrección atmosférica por el método Dark Object Subtraction (DOS1)  
Landsat 9 — Collection 2 Level-1
</h1>

<p>
Este cuaderno fue desarrollado como material docente para la asignatura <b>Teledetección</b>.
</p>

<b>Autor</b><br>

Julián Garzón Barrero<br>
Docente<br>
Programa de Ingeniería Topográfica y Geomática<br>
Universidad del Quindío<br>
juliangarzonb@uniquindio.edu.co

<hr>

<b>Apoyo técnico</b><br>

Estructuración y revisión computacional asistida mediante herramientas de inteligencia artificial (ChatGPT, OpenAI).

<hr>

<b>Propósito</b>

<p>
Presentar una implementación reproducible y documentada del método
<b>Dark Object Subtraction (DOS1)</b> para la corrección atmosférica de imágenes Landsat
y su posterior análisis radiométrico. El cuaderno también busca fomentar el
<b>pensamiento computacional</b>, abordando la corrección atmosférica como un problema
de modelado en el que los procesos físicos de interacción radiación–atmósfera–superficie,
se traducen en una secuencia explícita de operaciones matemáticas y computacionales
reproducibles.
</p>

</div>

57.2026

---

## 📘 Módulo 1 -- Autenticación e inicialización

**Objetivo**  
Conectar Google Colab con Google Earth Engine e inicializar el entorno de trabajo mediante un proyecto de Google Cloud habilitado para Earth Engine.

**Requisito previo**  
Cada estudiante debe disponer de:

- una cuenta de Google;
- acceso a Google Earth Engine;
- un proyecto de Google Cloud habilitado y registrado para utilizar Earth Engine;
- el identificador (`Project ID`) de dicho proyecto.

> **Importante:** el `Project ID` no corresponde al nombre visible del proyecto ni al número del proyecto. Debe utilizarse exactamente el identificador asignado en Google Cloud.

**Qué hace este módulo**

- instala `geemap`, que se utilizará posteriormente para la visualización interactiva;
- solicita el `Project ID` del estudiante;
- autentica la cuenta de Google utilizada en Colab;
- inicializa Earth Engine utilizando ese proyecto;
- realiza una comprobación mínima de la conexión.

**Resultado esperado**  
Earth Engine queda autenticado e inicializado y el notebook puede acceder tanto al catálogo público de Landsat como a los assets compartidos para esta práctica.

Si se reinicia el entorno de ejecución de Google Colab, este módulo deberá ejecutarse nuevamente.


In [ ]:
# ============================================================
# MÓDULO 1 -- Autenticación e inicialización de Earth Engine
# ============================================================

# ------------------------------------------------------------
# 0) Instalación de dependencias
# ------------------------------------------------------------

!pip -q install geemap

import ee


# ------------------------------------------------------------
# 1) Proyecto de Google Cloud
# ------------------------------------------------------------

PROJECT_ID = input(
    "Ingrese el Project ID de su proyecto de Google Cloud "
    "habilitado para Earth Engine: "
).strip()

if not PROJECT_ID:
    raise ValueError(
        "Debe ingresar un Project ID válido para utilizar Earth Engine."
    )


# ------------------------------------------------------------
# 2) Autenticación
# ------------------------------------------------------------

print("\nAutenticando Google Earth Engine...")

try:
    ee.Authenticate(auth_mode='notebook')

except Exception as e:
    print("\nNo fue posible completar la autenticación.")
    print("Verifique que:")
    print("  1. Haya iniciado sesión con una cuenta de Google válida.")
    print("  2. La cuenta tenga acceso a Google Earth Engine.")
    print("  3. Complete correctamente el proceso de autorización.")
    raise e


# ------------------------------------------------------------
# 3) Inicialización de Earth Engine
# ------------------------------------------------------------

print("\nInicializando Google Earth Engine...")

try:
    ee.Initialize(project=PROJECT_ID)

except Exception as e:
    print("\nNo fue posible inicializar Earth Engine.")
    print("Verifique que:")
    print("  1. El Project ID sea correcto.")
    print("  2. El proyecto esté habilitado para utilizar Earth Engine.")
    print("  3. La API de Earth Engine esté habilitada.")
    print("  4. La cuenta autenticada tenga acceso al proyecto.")
    raise e


# ------------------------------------------------------------
# 4) Verificación mínima de conexión
# ------------------------------------------------------------

try:
    test = ee.Number(1).getInfo()

    print("\n--------------------------------------------")
    print("Earth Engine inicializado correctamente")
    print("Proyecto activo:", PROJECT_ID)
    print("Prueba de conexión:", test)
    print("--------------------------------------------")

except Exception as e:
    print("\nEarth Engine se inicializó, pero no fue posible")
    print("completar la prueba de conexión.")
    raise e

---

## 📘 Módulo 2 -- Parámetros de entrada y verificación mínima

**Objetivo**  
Definir los insumos que vienen desde GEE y comprobar que Colab puede acceder correctamente a ellos.

**Entradas**  
- `IMAGE_ID`: escena Landsat 9 a procesar;  
- `AOI_ASSET`: área de interés;  
- `DARK_ASSET`: polígonos de objetos oscuros.

**Qué hace**  
- registra las rutas de entrada;  
- carga la imagen y las colecciones;  
- verifica acceso a la escena y a los assets.

**Resultado esperado**  
Se confirma que la imagen, el AOI y los objetos oscuros están disponibles para el procesamiento.

In [ ]:
# ============================================================
# MÓDULO 2 -- Parámetros de entrada y verificación mínima
# ============================================================

IMAGE_ID   = 'LANDSAT/LC09/C02/T1/LC09_009052_20250114'
AOI_ASSET  = 'users/juliangarzonb/RS/aoi_barranquilla'
DARK_ASSET = 'users/juliangarzonb/RS/objetos_oscuros_barranquilla'

print("IMAGE_ID  :", IMAGE_ID)
print("AOI_ASSET :", AOI_ASSET)
print("DARK_ASSET:", DARK_ASSET)

img  = ee.Image(IMAGE_ID)
aoi  = ee.FeatureCollection(AOI_ASSET)
dark = ee.FeatureCollection(DARK_ASSET)

print({
    "image_system_id": img.get("system:id").getInfo(),
    "aoi_n": aoi.size().getInfo(),
    "dark_n": dark.size().getInfo()
})

---

## 📘 Módulo 3 -- Lectura radiométrica

**Objetivo**  
Recuperar los metadatos radiométricos y solares de la escena Landsat 9 necesarios para iniciar la conversión de valores digitales a radiancia.

**Entradas**  
- imagen Landsat 9 Collection 2 Level-1 (`IMAGE_ID`);
- metadatos solares de la escena;
- coeficientes `RADIANCE_MULT_BAND_n` y `RADIANCE_ADD_BAND_n` para las bandas B2–B7.

**Qué hace**  
- lee la elevación solar, el acimut solar y la distancia Tierra-Sol;
- recupera la fecha y hora de adquisición de la escena;
- extrae los coeficientes multiplicativo y aditivo de radiancia por banda;
- verifica que los metadatos requeridos existan antes de continuar.

**Resultado esperado**  
Quedan disponibles los parámetros radiométricos necesarios para convertir los valores digitales de las bandas ópticas a radiancia.

In [ ]:
# ============================================================
# MÓDULO 3 -- Lectura radiométrica
# (Metadatos solares + fecha/hora escena + MULT/ADD radiancia)
# Landsat 9 C2 L1 — Lλ = MULT * DN + ADD
# ============================================================

import ee
from datetime import datetime

# ------------------------------------------------------------
# Utilidad — fail-fast (metadato obligatorio)
# ------------------------------------------------------------
def get_required(prop_name: str):
    v = img.get(prop_name).getInfo()
    if v is None:
        raise ValueError(f"Falta metadato requerido en la escena: {prop_name}")
    return v

def get_required_number(prop_name: str) -> ee.Number:
    return ee.Number(get_required(prop_name))

# ------------------------------------------------------------
# 1) Metadatos solares
# ------------------------------------------------------------
sun_elev = get_required('SUN_ELEVATION')
sun_az   = get_required('SUN_AZIMUTH')
d_es     = get_required('EARTH_SUN_DISTANCE')

print("SUN_ELEVATION:", sun_elev)
print("SUN_AZIMUTH:",   sun_az)
print("EARTH_SUN_DISTANCE:", d_es)

# ------------------------------------------------------------
# 2) Fecha y hora de captura (escena Landsat)
# ------------------------------------------------------------
date_acq   = get_required('DATE_ACQUIRED')        # YYYY-MM-DD
scene_time = get_required('SCENE_CENTER_TIME')    # HH:MM:SS.sssZ

print("DATE_ACQUIRED:", date_acq)
print("SCENE_CENTER_TIME (UTC):", scene_time)

# ------------------------------------------------------------
# 3) MULT / ADD radiancia (Collection 2): Lλ = MULT * DN + ADD
# ------------------------------------------------------------
bands = ['B2','B3','B4','B5','B6','B7']

mult = {b: get_required_number(f'RADIANCE_MULT_BAND_{b[1:]}') for b in bands}
add  = {b: get_required_number(f'RADIANCE_ADD_BAND_{b[1:]}')  for b in bands}

mult_info = {k: v.getInfo() for k, v in mult.items()}
add_info  = {k: v.getInfo() for k, v in add.items()}

print("RADIANCE_MULT:", mult_info)
print("RADIANCE_ADD :", add_info)

---

## 📘 Módulo 4 -- Conversión DN → radiancia

**Objetivo**  
Convertir los valores digitales de las bandas ópticas B2–B7 a radiancia espectral usando los coeficientes radiométricos de la escena Landsat 9.

**Entradas**  
- imagen Landsat recortada al AOI (`img_clip`);
- coeficientes `RADIANCE_MULT_BAND_n` y `RADIANCE_ADD_BAND_n`;
- bandas ópticas B2, B3, B4, B5, B6 y B7.

**Procesos**  
Para cada banda, aplica la relación:

$$
L_{\lambda} = M_L \cdot Q_{cal} + A_L
$$

donde:

- $L_{\lambda}$ : radiancia espectral   
- $Q_{cal}$ : valor digital ($DN$)
- $M_L$ : coeficientes radiométricos multiplicativos del sensor
- $A_L$ : coeficientes radiométricos aditivos del sensor

Luego organiza los resultados en una sola imagen multibanda de radiancia.

**Salida**  
Se obtiene un stack de radiancia (`L_img`) para las bandas B2–B7, listo para la estimación posterior de la radiancia de trayectoria y la corrección atmosférica DOS1.

---

### Papel dentro del flujo DOS1

Este módulo completa la reconstrucción radiométrica del experimento.

A partir de la radiancia es posible:

* calcular percentiles oscuros
* estimar la radiancia de trayectoria ($L_p$)
* aplicar la corrección DOS1

Marca el paso desde la calibración del sensor hacia la estimación atmosférica.

In [ ]:
# ============================================================
# MÓDULO 4 -- Conversión DN → radiancia
# (Landsat 9 C2 L1: Lλ = ML * DN + AL)
# ============================================================

# Recorte de la imagen
geom = aoi.geometry()
img_clip = img.clip(geom)

bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7']

radiance = {}

for b in bands:
    # Coeficientes radiométricos (Collection 2)
    ML = ee.Number(img.get(f'RADIANCE_MULT_BAND_{b[1:]}'))
    AL = ee.Number(img.get(f'RADIANCE_ADD_BAND_{b[1:]}'))

    # DN → radiancia (forzar float para evitar ambigüedad de tipos)
    DN = img_clip.select(b).toFloat()
    L  = DN.multiply(ML).add(AL).rename(b)

    radiance[b] = L # guarda la imagen de radiancia de la banda dentro del diccionario de radiancia

# Stack radiancia en un solo ee.Image (bandas B2..B7)
L_img = ee.Image.cat([radiance[b] for b in bands]).rename(bands)

print("Radiancia stack listo")
print("Bandas radiancia:", L_img.bandNames().getInfo())

---

## 📘 Módulo 5 -- Estimación de $L_{min}$ (percentil dark)



**Objetivo**  
Estimar la radiancia de los píxeles más oscuros dentro de los polígonos "DARK_ASSET" mediante un percentil bajo de la distribución de radiancia.

En lugar de usar el valor mínimo absoluto —que suele ser inestable por errores del sensor—, usamos el percentil 1%. Este funciona como un promedio de los valores más bajos, capturando la esencia de la oscuridad profunda sin verse afectado por píxeles erróneos o valores extremos aislados.

Este valor, llamado **$L_{min}$**, representa la señal residual registrada por el sensor en superficies cuya reflectancia es cercana a cero y constituye la base empírica para derivar la radiancia de trayectoria en el método DOS1.


**Entradas**  
- imagen de radiancia (`L_img`);
- polígonos de objetos oscuros (`dark`);
- área de interés (`aoi`);
- percentil bajo de la distribución de radiancia `(p = 1%)`.

**Procesos**  
Primero define como dominio estadístico la intersección entre los objetos oscuros y el AOI.  
Luego calcula, para cada banda, el percentil 1% de la radiancia dentro de ese dominio.

No se usa el valor mínimo absoluto, porque puede verse afectado por ruido o valores atípicos. En su lugar, el percentil bajo ofrece una estimación más robusta de la señal oscura observada.

**Salidas**  
Se obtienen los valores de $L_{min}$ por banda, que servirán como base empírica para estimar posteriormente la radiancia de trayectoria $(L_p)$ en el método DOS1.

In [ ]:
# ============================================================
# MÓDULO 5 -- Estimación Lmin (percentil dark)
# ============================================================

bands = ['B2','B3','B4','B5','B6', 'B7']

# Dominio estadístico: objetos oscuros recortados al AOI
dark_region = dark.geometry().intersection(aoi.geometry(), 1)

# Percentil bajo (DOS1)
p = 1
reducer = ee.Reducer.percentile([p])

Lmin_dict = L_img.reduceRegion(
    reducer=reducer,
    geometry=dark_region,
    scale=30,
    maxPixels=1e9
).getInfo()

print(f"Resultados Lmin (p={p}%):")
for k, v in Lmin_dict.items():
    print(k, ":", v)

---

## 📘 Módulo 6 -- $L_{DO1\%}$ (radiancia del objeto oscuro equivalente a 1 % de reflectancia)

### Propósito

Calcular, para cada banda, la radiancia teórica asociada a una superficie oscura con reflectancia del 1 % (**$L_{DO1\%}$**), usada como referencia física en el método DOS1.

---

### Fundamento físico

El método DOS1 asume que los objetos más oscuros de la escena no tienen reflectancia cero, sino una reflectancia mínima aproximada de ($ρ = 0.01$).

Para Landsat 8 y 9, este valor puede obtenerse imponiendo esa reflectancia en la ecuación oficial de reflectancia TOA del producto y convirtiendo luego el resultado a radiancia:


$$
L_{DO1\%}
=
M_L\left(\frac{0.01\,\cos\theta_z-A_\rho}{M_\rho}\right)+A_L
$$

donde:

- $M_L$ : RADIANCE_MULT (metadato Landsat)  
- $A_L$ : RADIANCE_ADD (metadato Landsat)  
- $M_\rho$ : REFLECTANCE_MULT (metadato Landsat)  
- $A_\rho$ : REFLECTANCE_ADD (metadato Landsat)
- $\theta_z$: ángulo cenital solar

---

### Entradas

* elevación solar (SUN_ELEVATION), convertida a ángulo cenital
* RADIANCE_MULT / RADIANCE_ADD  
* REFLECTANCE_MULT / REFLECTANCE_ADD  
* bandas ópticas B2–B7.

---

### Procesos

1. Conversión de elevación solar a ángulo cenital.  
2. Cálculo del coseno del ángulo cenital solar.  
3. Imposición de una reflectancia mínima de 1 % en la ecuación TOA del producto.  
4. Conversión del valor resultante a radiancia por banda.  
5. Obtiene por cada banda, la radiancia equivalente $L_{DO1\%}$.


### Salidas

* Valores de **$L_{DO1\%}$** por banda, que servirán como referencia teórica para estimar la radiancia de trayectoria ($L_p$).


In [ ]:
# ============================================================
# MÓDULO 6 -- L_DO1% (radiancia equivalente a 1% reflectancia)
# Landsat 8/9 (C2 L1) — sin ESUN, usando coeficientes del producto
#
# Idea:
#  1) imponer ρ = 0.01 en la ecuación TOA de reflectancia del producto
#  2) despejar DN_DO
#  3) convertir ese DN_DO a radiancia con MULT/ADD de radiancia
# ============================================================

import math

bands = ['B2','B3','B4','B5','B6','B7']

# ------------------------------------------------------------
# 1) Geometría solar: cos(θz)
# ------------------------------------------------------------
theta_z = 90.0 - sun_elev                        # θz = 90° - SUN_ELEVATION
cos_tz  = math.cos(math.radians(theta_z))        # cos(θz) = sin(SUN_ELEVATION)

# Reflectancia mínima asumida (DOS1)
rho_do = 0.01

# ------------------------------------------------------------
# 2) Leer coeficientes del producto (por banda)
#    - Reflectancia: ρ = (Mρ*DN + Aρ) / cosθz
#    - Radiancia:    L = ML*DN + AL
# ------------------------------------------------------------
def get_required(prop_name: str):
    v = img.get(prop_name).getInfo()
    if v is None:
        raise ValueError(f"Falta metadato requerido: {prop_name}")
    return float(v)

M_rho = {b: get_required(f'REFLECTANCE_MULT_BAND_{b[1:]}') for b in bands}
A_rho = {b: get_required(f'REFLECTANCE_ADD_BAND_{b[1:]}')  for b in bands}

M_L   = {b: get_required(f'RADIANCE_MULT_BAND_{b[1:]}')    for b in bands}
A_L   = {b: get_required(f'RADIANCE_ADD_BAND_{b[1:]}')     for b in bands}

# ------------------------------------------------------------
# 3) DN_DO y L_DO1%
#    ρ = (Mρ*DN + Aρ)/cosθz  =>  DN_DO = (ρ*cosθz - Aρ)/Mρ
#    L_DO1% = ML*DN_DO + AL
# ------------------------------------------------------------
DN_DO = {}
L_DO  = {}

for b in bands:
    DN_DO[b] = (rho_do * cos_tz - A_rho[b]) / M_rho[b]
    L_DO[b]  = (M_L[b] * DN_DO[b]) + A_L[b]

print("L_DO1% (ρ=0.01):")
for b in bands:
    print(b, ":", L_DO[b])

---

## 📘 Módulo 7 -- Cálculo de radiancia de trayectoria $(L_p)$

**Objetivo**  
Estimar, para cada banda, la radiancia de trayectoria $(L_p)$, entendida como la contribución atmosférica aditiva que el método DOS1 busca remover.

**Fundamento**  
En los objetos más oscuros de la escena, la radiancia observada $(L_{min})$ puede interpretarse como la suma de una señal superficial mínima y de la contribución atmosférica.

Bajo el supuesto DOS1, la radiancia de trayectoria se estima como:

$$
L_p = L_{min} - L_{DO1\%}
$$

donde $L_{min}$ es la radiancia mínima estimada empíricamente y $L_{DO1\%}$ la radiancia teórica equivalente a una reflectancia del 1 %.

**Entradas**  
- valores de $L_{min}$ por banda;
- valores de $L_{DO1\%}$ por banda;
- bandas ópticas B2–B7.

**Proceso**  
Calcula, para cada banda, la diferencia entre la radiancia mínima observada y la radiancia oscura teórica.

**Salida**  
Se obtienen los valores de $L_p$ por banda, que se usarán como término aditivo atmosférico en la corrección DOS1.


In [ ]:
# ============================================================
# MÓDULO 7 -- Radiancia de trayectoria Lp
# ============================================================

Lp_final = {}

for b in bands:
    Lmin = Lmin_dict[b]
    Lp_final[b] = Lmin - L_DO[b]

print("Lp:")
for b,v in Lp_final.items():
    print(b,":",v)

---

## 📘 Módulo 8 -- Aplicación de DOS1 en radiancia

**Objetivo**  
Corregir la radiancia a nivel sensor mediante la sustracción de la radiancia de trayectoria $(L_p)$ estimada para cada banda.

**Fundamento**  
En el esquema DOS1, la radiancia observada por el sensor puede expresarse como la suma de una contribución superficial y una contribución atmosférica aditiva:

$$
L_{\lambda}^{sen} = L_{\lambda}^{sup} + L_p
$$

Por tanto, la radiancia corregida se obtiene como:

$$
L_{\lambda}^{corr} = L_{\lambda}^{sen} - L_p
$$

Para evitar valores físicamente no válidos, se impone además:

$$
L_{\lambda}^{corr} =
\max\left(L_{\lambda}^{sen} - L_p,\; 0\right)
$$

**Entradas**  
- imagen de radiancia a nivel sensor (`L_img`);
- valores de radiancia de trayectoria (`Lp_final`) por banda.

**Proceso**  
- resta $(L_p$) a cada banda de radiancia;
- fuerza a cero los valores negativos;
- reconstruye la imagen multibanda de radiancia corregida.

**Salida**  
Se obtiene una imagen de radiancia corregida (`Lcorr_img`), lista para la conversión posterior a reflectancia corregida por DOS1.

---

In [ ]:
# ============================================================
# MÓDULO 8 -- Aplicar DOS1 en radiancia
# ============================================================
# Lcorr = max(L_lambda_sen - Lp, 0)
# Radiancia corregida por sustracción de la radiancia de trayectoria

Lcorr = {}

for b in bands:
    Lp_b = ee.Number(Lp_final[b])  # escalar server-side
    Lcorr[b] = L_img.select(b).subtract(Lp_b).max(0).rename(b)

Lcorr_img = ee.Image.cat([Lcorr[b] for b in bands]).rename(bands)

print("Radiancia corregida por DOS1 lista")
print("Bandas:", Lcorr_img.bandNames().getInfo())

---

## 📘 Módulo 9 -- Conversión de radiancia DOS1 a reflectancia

**Objetivo**  
Convertir la radiancia corregida por DOS1 a reflectancia TOA corregida, usando los coeficientes oficiales del producto Landsat 9. <br>



**Fundamento**  
Para Landsat 8/9, la reflectancia TOA y la radiancia se relacionan con los coeficientes radiométricos del producto. Si primero se corrige la radiancia por sustracción de la radiancia de trayectoria, esa radiancia corregida puede transformarse a reflectancia mediante:<br><br>


$$
\rho_{DOS1}=
\frac{M_{\rho}}{\cos \theta_z}
\cdot
\left(
\frac{L_{\lambda}^{corr}-A_L}{M_L}
\right)
+
\frac{A_{\rho}}{\cos \theta_z}
$$
<br>
donde $M_{\rho}$ y $A_{\rho}$ son los coeficientes de reflectancia, $M_L$ y $A_L$ los coeficientes de radiancia, y $\theta_z$ el ángulo cenital solar.

<br>

**Entradas**  
- imagen de radiancia corregida (`Lcorr_img`);
- coeficientes de radiancia y reflectancia por banda;
- geometría solar de la escena.

<br>

**Proceso**  
- recupera los coeficientes de reflectancia;
- calcula $\cos(\theta_z)$;
- convierte la radiancia corregida a reflectancia banda a banda;
- aplica un control físico para mantener los valores en el rango \([0,1]\).

<br>

**Salida**  
Se obtiene una imagen multibanda de reflectancia corregida por DOS1 (`rho_dos1_img`), lista para visualización, análisis espectral y cálculo de índices.

<br>

**Observación**  
Esta reflectancia sigue siendo una aproximación simplificada, porque DOS1 corrige principalmente la componente atmosférica aditiva y no modela explícitamente otros efectos atmosféricos.

In [ ]:
# ============================================================
# MÓDULO 9 -- Conversión de radiancia DOS1 a reflectancia
# ============================================================
# ρ_DOS1 = M_ρ / cos(θ_z) * ((Lcorr - A_L) / M_L) + A_ρ / cos(θ_z)

import ee
import math

# ------------------------------------------------------------
# 1) Coeficientes de reflectancia desde metadatos de la escena
# ------------------------------------------------------------
refl_mult = {b: get_required_number(f'REFLECTANCE_MULT_BAND_{b[1:]}') for b in bands}
refl_add  = {b: get_required_number(f'REFLECTANCE_ADD_BAND_{b[1:]}')  for b in bands}

refl_mult_info = {k: v.getInfo() for k, v in refl_mult.items()}
refl_add_info  = {k: v.getInfo() for k, v in refl_add.items()}

print("REFLECTANCE_MULT:", refl_mult_info)
print("REFLECTANCE_ADD :", refl_add_info)

# ------------------------------------------------------------
# 2) Geometría solar
# ------------------------------------------------------------
theta_z_deg = 90.0 - sun_elev
cos_theta_z = ee.Number(math.cos(math.radians(theta_z_deg)))

print("theta_z (deg):", theta_z_deg)
print("cos(theta_z):", cos_theta_z.getInfo())

# ------------------------------------------------------------
# 3) Conversión banda a banda
# ------------------------------------------------------------
rho_dos1 = {}

for b in bands:
    M_L   = mult[b]
    A_L   = add[b]
    M_rho = refl_mult[b]
    A_rho = refl_add[b]

    rho_b = (
        Lcorr_img.select(b)
        .subtract(A_L)
        .divide(M_L)
        .multiply(M_rho)
        .divide(cos_theta_z)
        .add(A_rho.divide(cos_theta_z))
        .max(0)
        .rename(b)
    )

    rho_dos1[b] = rho_b

# ------------------------------------------------------------
# 4) Reconstrucción de imagen multibanda
# ------------------------------------------------------------
rho_dos1_img = ee.Image.cat([rho_dos1[b] for b in bands]).rename(bands)

# ------------------------------------------------------------
# 5) Control físico opcional: recorte a [0,1]
# ------------------------------------------------------------
rho_dos1_img = rho_dos1_img.max(0).min(1)

print("Reflectancia corregida por DOS1 lista")
print("Bandas:", rho_dos1_img.bandNames().getInfo())

---

## 📘 Módulo 10 -- Visualización comparativa (TOA vs DOS1)

### Propósito

Realizar una verificación visual del efecto de la corrección atmosférica DOS1 mediante la comparación directa entre la radiancia TOA y la radiancia corregida.

### Enfoque

Se construye una visualización RGB (4-3-2) empleando un estiramiento robusto basado en percentiles (p2–p98) calculados sobre la imagen corregida.
Este criterio garantiza una visualización estable y evita que valores extremos condicionen el contraste.

Ambas imágenes se muestran con el mismo rango radiométrico, permitiendo evaluar el efecto aditivo de la corrección.

### Interpretación

La imagen corregida debe presentar:

* reducción del velo atmosférico
* mayor contraste en superficies oscuras
* recuperación de gradientes radiométricos en agua y sombra

La comparación mantiene constante la geometría y el stretch, por lo que las diferencias observadas se atribuyen exclusivamente a la corrección DOS1.

### Resultado esperado

Una escena visualmente más “limpia”, con disminución del sesgo atmosférico sin alteración de la estructura espacial.

Este módulo constituye el control visual del comportamiento físico de la corrección.


In [ ]:
# ============================================================
# MÓDULO 10 — Visualización comparativa (Reflectancia TOA vs DOS1)
# ============================================================
# Comparación visual entre:
# - rho_toa_img   : reflectancia TOA
# - rho_dos1_img  : reflectancia corregida por DOS1
#
# El stretch RGB (4-3-2) se calcula sobre rho_dos1_img
# usando percentiles p2-p98 y se aplica igual a ambas imágenes.
# ============================================================

import ee
import geemap

# ------------------------------------------------------------
# 1) Verificación mínima de variables requeridas
# ------------------------------------------------------------
required_vars = ['img', 'bands', 'rho_dos1_img', 'refl_mult', 'refl_add', 'cos_theta_z']
for v in required_vars:
    if v not in globals():
        raise RuntimeError(f"Falta la variable requerida: {v}")

# Si no existe aoi, usar la geometría de la imagen
if 'aoi' not in globals():
    aoi = img.geometry()
    print("No se encontró 'aoi'. Se usará la geometría de la imagen.")

# ------------------------------------------------------------
# 2) Construcción de reflectancia TOA
#    ρ_TOA = (Mρ * DN + Aρ) / cos(θz)
# ------------------------------------------------------------
rho_toa = {}

for b in bands:
    M_rho = refl_mult[b]
    A_rho = refl_add[b]

    rho_b = (
        img.select(b)
        .multiply(M_rho)
        .add(A_rho)
        .divide(cos_theta_z)
        .max(0)
        .rename(b)
    )

    rho_toa[b] = rho_b

rho_toa_img = ee.Image.cat([rho_toa[b] for b in bands]).rename(bands)

# Control físico opcional: recorte a [0,1]
rho_toa_img = rho_toa_img.max(0).min(1)

print("Reflectancia TOA lista")
print("Bandas:", rho_toa_img.bandNames().getInfo())

# ------------------------------------------------------------
# 3) Cálculo de percentiles p2-p98 sobre la imagen DOS1
#    para RGB = B4-B3-B2
# ------------------------------------------------------------
rgb_bands = ['B4', 'B3', 'B2']

pct = rho_dos1_img.select(rgb_bands).reduceRegion(
    reducer=ee.Reducer.percentile([2, 98]),
    geometry=aoi,
    scale=30,
    maxPixels=1e13,
    bestEffort=True
)

pct_info = pct.getInfo()

vis_params = {
    'bands': rgb_bands,
    'min': [
        pct_info['B4_p2'],
        pct_info['B3_p2'],
        pct_info['B2_p2']
    ],
    'max': [
        pct_info['B4_p98'],
        pct_info['B3_p98'],
        pct_info['B2_p98']
    ],
    'gamma': 1.0
}

print("Parámetros de visualización RGB (p2-p98):")
print(vis_params)

# ------------------------------------------------------------
# 4) Mapa comparativo TOA vs DOS1
# ------------------------------------------------------------

rho_toa_clip  = rho_toa_img.clip(aoi)
rho_dos1_clip = rho_dos1_img.clip(aoi)

Map = geemap.Map(basemap="HYBRID")

toa_layer = geemap.ee_tile_layer(
    rho_toa_clip,
    vis_params,
    "Reflectancia TOA"
)

dos1_layer = geemap.ee_tile_layer(
    rho_dos1_clip,
    vis_params,
    "Reflectancia DOS1"
)

Map.split_map(
    left_layer=toa_layer,
    right_layer=dos1_layer,
    left_label="TOA",
    right_label="DOS1"
)

Map.centerObject(aoi, 11)

display(Map)

---

## 📘 Módulo 11 -- Histogramas por banda (Reflectancia TOA vs Reflectancia DOS1)

### Propósito

Evaluar el efecto radiométrico de la corrección atmosférica DOS1 mediante la comparación de las distribuciones de reflectancia antes (TOA) y después de la corrección.

### Enfoque

Para cada banda óptica se calculan histogramas dentro del AOI utilizando un rango robusto definido por percentiles (p2–p98).  
Este criterio evita que valores extremos condicionen la visualización de la distribución.

Los histogramas se normalizan a frecuencia relativa para permitir la comparación directa entre la reflectancia aparente TOA (`ρ_TOA`) y la reflectancia corregida por DOS1 (`ρ_DOS1`).

### Interpretación

La corrección DOS1 elimina la componente atmosférica aditiva estimada mediante la radiancia de trayectoria.  
Como consecuencia, la distribución de reflectancia suele mostrar:

* reducción del sesgo introducido por la dispersión atmosférica
* mayor concentración de valores en rangos físicamente plausibles
* mejor diferenciación entre superficies oscuras y brillantes

La forma general del histograma debería mantenerse, lo que indica que la corrección actúa principalmente sobre el sesgo atmosférico sin alterar la estructura espectral de la escena.

### Resultado esperado

* Ajuste de la distribución radiométrica respecto a la reflectancia TOA.
* Mayor contraste entre clases superficiales.
* Conservación de la variabilidad interna de la escena.

Este módulo constituye una verificación estadística del comportamiento de la corrección atmosférica aplicada.

In [ ]:
# ============================================================
# MÓDULO 11 — Histogramas por banda (Reflectancia TOA vs DOS1)
# ============================================================

import ee
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1) Verificación mínima
# ------------------------------------------------------------
required_vars = ['rho_toa_img', 'rho_dos1_img', 'bands']
for v in required_vars:
    if v not in globals():
        raise RuntimeError(f"Falta la variable requerida: {v}")

if 'aoi' not in globals():
    aoi = img.geometry()
    print("No se encontró 'aoi'. Se usará la geometría de la imagen.")

# ------------------------------------------------------------
# 2) Función para extraer valores de una banda dentro del AOI
# ------------------------------------------------------------
def get_band_values(image, band, region, scale=30):
    values = image.select(band).reduceRegion(
        reducer=ee.Reducer.toList(),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True
    ).get(band)

    values = ee.List(values).getInfo()
    values = np.array(values, dtype=np.float32)
    values = values[np.isfinite(values)]

    return values

# ------------------------------------------------------------
# 3) Configuración de bandas
# ------------------------------------------------------------
band_labels = {
    'B2': 'B2 (Blue)',
    'B3': 'B3 (Green)',
    'B4': 'B4 (Red)',
    'B5': 'B5 (NIR)',
    'B6': 'B6 (SWIR1)',
    'B7': 'B7 (SWIR2)',
}

# ------------------------------------------------------------
# 4) Figura
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

# ------------------------------------------------------------
# 5) Histogramas banda a banda
# ------------------------------------------------------------
for ax, b in zip(axes, bands):

    print(f"Procesando {b}...")

    toa_vals  = get_band_values(rho_toa_img,  b, aoi, scale=30)
    dos1_vals = get_band_values(rho_dos1_img, b, aoi, scale=30)

    # Asegurar valores válidos
    toa_vals  = toa_vals[np.isfinite(toa_vals)]
    dos1_vals = dos1_vals[np.isfinite(dos1_vals)]

    if len(toa_vals) == 0 or len(dos1_vals) == 0:
        ax.set_title(f"{band_labels[b]}\nSin datos")
        ax.axis("off")
        continue

    # Rango robusto común basado en DOS1
    p2  = np.percentile(dos1_vals, 2)
    p98 = np.percentile(dos1_vals, 98)

    # Limitar valores al rango robusto
    toa_clip  = toa_vals[(toa_vals >= p2) & (toa_vals <= p98)]
    dos1_clip = dos1_vals[(dos1_vals >= p2) & (dos1_vals <= p98)]

    # Si el recorte deja sin datos, usar originales
    if len(toa_clip) == 0:
        toa_clip = toa_vals
    if len(dos1_clip) == 0:
        dos1_clip = dos1_vals

    # Histograma con frecuencia relativa
    bins = 80

    ax.hist(
        toa_clip,
        bins=bins,
        density=True,
        alpha=0.55,
        label='TOA'
    )

    ax.hist(
        dos1_clip,
        bins=bins,
        density=True,
        alpha=0.55,
        label='DOS1'
    )

    ax.set_title(band_labels[b])
    ax.set_xlabel('Reflectancia')
    ax.set_ylabel('Frecuencia relativa')
    ax.legend()

# ------------------------------------------------------------
# 6) Ajuste final
# ------------------------------------------------------------
plt.suptitle('Histogramas por banda — Reflectancia TOA vs Reflectancia DOS1', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 📊 Evaluación de la corrección DOS1 mediante histogramas de reflectancia

### Propósito

Evaluar el efecto radiométrico de la corrección atmosférica DOS1 comparando la distribución de reflectancia antes (TOA) y después de la corrección para cada banda óptica.

Los histogramas permiten analizar el comportamiento estadístico de la señal espectral y constituyen una evidencia empírica del impacto de la corrección atmosférica sobre la escena.

---

### Principio físico

En el modelo DOS1 la radiancia registrada por el sensor puede expresarse como:



$ L_{\lambda}^{TOA} = L_{\lambda}^{sup} + L_p $

donde \(L_p\) corresponde a la **radiancia de trayectoria**, producida por dispersión atmosférica.

La corrección DOS1 estima este término y lo sustrae de la radiancia observada.  
Posteriormente, la radiancia corregida se transforma a reflectancia utilizando la calibración radiométrica del sensor.

Por esta razón, la comparación de histogramas se realiza sobre **reflectancia aparente TOA $(\rho_{TOA})$** y **reflectancia corregida $(\rho_{DOS1})$**.

---

### Resultados observados

#### Bandas visibles (B2–B4)

Las diferencias entre TOA y DOS1 son más pronunciadas en las longitudes de onda cortas.

Se observa una redistribución de los valores de reflectancia tras la corrección, lo que indica la reducción del sesgo introducido por la dispersión atmosférica, particularmente en la banda azul, donde este efecto es dominante.

La corrección permite una mejor diferenciación entre superficies oscuras y brillantes, reflejando una recuperación parcial de la señal superficial.

---

#### Banda NIR (B5)

La corrección afecta principalmente a los valores bajos de reflectancia asociados a agua, sombras o superficies oscuras.

Los valores altos vinculados a vegetación cambian en menor medida, lo que sugiere que la corrección preserva el contraste biofísico de la escena y mantiene la estructura espectral de la señal.

---

#### Bandas SWIR (B6–B7)

Las diferencias entre TOA y DOS1 son menores, lo cual es consistente con la menor contribución de la dispersión atmosférica en longitudes de onda largas.

La estabilidad relativa de los histogramas en estas bandas constituye un indicador adicional de coherencia física del procedimiento de corrección.

---

### Interpretación radiométrica

La comparación entre histogramas refleja la reducción del sesgo atmosférico introducido por la radiancia de trayectoria.

Como consecuencia:

* disminuye el offset radiométrico introducido por la atmósfera
* mejora la separabilidad entre clases superficiales
* aumenta el contraste radiométrico efectivo
* se preserva la coherencia espectral entre bandas

Este comportamiento es consistente con el modelo físico subyacente al método DOS1.

---

### Evidencia metodológica

Los histogramas constituyen una verificación empírica del proceso de corrección porque:

1. evidencian el impacto radiométrico de la dispersión atmosférica en la escena,
2. muestran que la corrección actúa principalmente sobre el sesgo atmosférico,
3. permiten observar la recuperación del contraste radiométrico superficial,
4. confirman la estabilidad espectral del procedimiento.

---

### Conclusión

El análisis histogramático confirma que la corrección DOS1 reduce el sesgo atmosférico asociado a la radiancia de trayectoria de forma consistente con la física del sistema sensor–atmósfera–superficie.

La comparación entre las distribuciones de reflectancia TOA y DOS1 proporciona una evidencia estadística clara de la mejora radiométrica de la escena y respalda la utilización de la imagen corregida en análisis espectrales posteriores.

---

In [ ]:
# ============================================================
# EXPORTACIÓN — Reflectancia corregida DOS1 (rho_dos1_img)
# Descarga a Google Drive
# ============================================================

import ee

# Bandas ópticas Landsat 9
bands = ['B2','B3','B4','B5','B6','B7']

task = ee.batch.Export.image.toDrive(
    image = rho_dos1_img.select(bands),
    description = 'L9_DOS1_reflectance',
    folder = 'GEE_exports2',            # carpeta en Google Drive
    fileNamePrefix = 'L9_DOS1_reflectance',
    region = aoi.geometry(),
    scale = 30,
    maxPixels = 1e13,
    fileFormat = 'GeoTIFF'
)

task.start()

print("Exportación iniciada.")
print("Producto exportado: Reflectancia corregida DOS1")
print("Revise la pestaña Tasks o su Google Drive")